## Check Status - Verify Current State
Stop and Clean-up

In [0]:
# Title: Check Status - Verify Current State

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors.platform import NotFound

print(f"\n🔍 CHECKING DIGITAL TWIN APPLICATION STATUS")
print(f"{'='*80}\n")

w = WorkspaceClient()

# Define default parameters (in case loading fails)
APP_NAME = "digitaltwinapp-sk"
INSTANCE_NAME = "lakebasedigitaltwins-sk"
MAPPING_PIPELINE_NAME = "dt_sdp_pipeline"
BRONZE_TABLE = "sunny_catalog.default.dt_bronze"
TRIPLE_TABLE = "sunny_catalog.default.triples"
SYNCED_TABLE = "sunny_catalog.default.latest_sensor_triples"

# Try to load parameters from notebook
try:
    dbutils.notebook.run("./0-Parameters", 60)
    print("✅ Parameters loaded successfully\n")
except Exception as e:
    print(f"⚠️  Could not load parameters: {e}")
    print(f"   Using default parameter values\n")

# Check App Status
print("📱 Databricks App Status:")
print("-" * 80)
try:
    app = w.apps.get(APP_NAME)
    print(f"   Name: {app.name}")
    
    # Handle different SDK versions for status
    if hasattr(app, 'status'):
        status = app.status.state if hasattr(app.status, 'state') else str(app.status)
        print(f"   Status: {status}")
    elif hasattr(app, 'state'):
        print(f"   Status: {app.state}")
    else:
        print(f"   Status: Available (status field not accessible)")
    
    if hasattr(app, 'url') and app.url:
        print(f"   URL: {app.url}")
    
    print(f"   ✅ App exists and is accessible")
except NotFound:
    print(f"   ❌ App '{APP_NAME}' not found")
except Exception as e:
    print(f"   ❌ Error checking app: {e}")

# Check Lakebase Instance
print("\n🗄️  Lakebase Instance Status:")
print("-" * 80)
try:
    # Check if database attribute exists (newer SDK versions)
    if hasattr(w, 'database'):
        instance = w.database.get_database_instance(INSTANCE_NAME)
        print(f"   Name: {instance.name}")
        
        # Handle different attribute names for status
        if hasattr(instance, 'state'):
            print(f"   Status: {instance.state}")
        elif hasattr(instance, 'status'):
            print(f"   Status: {instance.status}")
        else:
            print(f"   Status: Active (status field not accessible)")
        
        if hasattr(instance, 'host'):
            print(f"   Host: {instance.host}")
        
        print(f"   ✅ Lakebase instance exists")
    else:
        print(f"   ⚠️  Lakebase API not available in current SDK version")
        print(f"   💡 Upgrade SDK: %pip install --upgrade databricks-sdk>=0.61.0")
        print(f"   Checking via SQL instead...")
        
        # Try to check synced table as proxy for Lakebase existence
        try:
            spark.sql(f"DESCRIBE TABLE {SYNCED_TABLE}")
            print(f"   ✅ Synced table exists (Lakebase likely operational)")
        except:
            print(f"   ❌ Cannot verify Lakebase status")
except NotFound:
    print(f"   ❌ Instance '{INSTANCE_NAME}' not found")
except Exception as e:
    print(f"   ❌ Error checking instance: {e}")

# Check DLT Pipeline
print("\n🔄 DLT Pipeline Status:")
print("-" * 80)
try:
    pipeline = next(w.pipelines.list_pipelines(filter=f"name LIKE '{MAPPING_PIPELINE_NAME}'"))
    print(f"   Name: {pipeline.name}")
    print(f"   Pipeline ID: {pipeline.pipeline_id}")
    print(f"   ✅ Pipeline exists")
except StopIteration:
    print(f"   ❌ Pipeline '{MAPPING_PIPELINE_NAME}' not found")
except Exception as e:
    print(f"   ❌ Error checking pipeline: {e}")

# Check Tables
print("\n📊 Unity Catalog Tables:")
print("-" * 80)

tables_to_check = [
    ("Bronze Table", BRONZE_TABLE),
    ("Triples Table", TRIPLE_TABLE),
    ("Synced Table", SYNCED_TABLE)
]

for table_name, table_path in tables_to_check:
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_path}").collect()[0]['cnt']
        print(f"   {table_name}: ✅ Exists ({count:,} rows)")
    except Exception as e:
        print(f"   {table_name}: ❌ Not found or error - {str(e)[:50]}")

print(f"\n{'='*80}")
print("Status check complete!")